In [9]:
import json
import pandas as pd
from tqdm import tqdm
import requests


# URL с данными
url = "https://orcid.org/0000-0002-8520-7267/allWorks.json?sort=date&sortAsc=false"
print("Загружаем данные...")
response = requests.get(url, headers={"Accept": "application/json"})

if response.status_code != 200:
    print(f"Ошибка загрузки: статус {response.status_code}")
    exit()

# Преобразуем JSON в словарь Python
data = response.json()
print("Данные успешно загружены.")





Загружаем данные...
Данные успешно загружены.


In [11]:
results = []

# Проходим по всем группам
for group in tqdm(data.get('groups', []), desc="Обработка групп"):
    works = group.get('works', [])
    for work in works:
        # Год публикации
        pub_date = work.get('publicationDate', {})
        year = pub_date.get('year') if pub_date else None
        
        # Название
        title_obj = work.get('title', {})
        title = title_obj.get('value') if title_obj else None
        
        # Ищем DOI и EID в workExternalIdentifiers
        doi = None
        eid = None
        ext_ids = work.get('workExternalIdentifiers', [])
        for ext in ext_ids:
            id_type = ext.get('externalIdentifierType', {}).get('value')
            if id_type == 'doi':
                doi = ext.get('externalIdentifierId', {}).get('value')
            elif id_type == 'eid':
                eid = ext.get('externalIdentifierId', {}).get('value')
        
        results.append({
            'year': year,
            'title': title,
            'doi': doi,
            'eid': eid
        })

# Сохраняем в JSON
with open('parsed_works.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Сохраняем в CSV (для удобства просмотра)
df = pd.DataFrame(results)
df.to_csv('parsed_works.csv', index=False, encoding='utf-8-sig')

print(f"Обработано записей: {len(results)}")

Обработка групп: 100%|████████████████████████████████████████████████████████████| 713/713 [00:00<00:00, 80293.70it/s]

Обработано записей: 1195


In [25]:
df.head(20)

,year,title,doi,eid
0,2026,Deep Unfolding of Iterative Precoding-Based Nu...,10.1109/TCOMM.2025.3642692,None
1,2026,Resource Allocation Schemes for Scalable Panel...,10.1109/OJVT.2026.3652908,None
2,2026,Towards Incorporating mmWave Beam Tracking for...,10.1109/ACCESS.2026.3667877,None
3,2025,On Deep Learning Hybrid Architectures for MIMO...,10.3390/electronics14234692,None
4,2025,On Deep Learning Hybrid Architectures for MIMO...,10.3390/electronics14234692,None
5,2025,On Deep Learning Hybrid Architectures for MIMO...,10.3390/electronics14234692,2-s2.0-105024537957
6,2025,Data-Oriented Channel Knowledge Map IoT Transm...,10.1109/VTC2025-Spring65109.2025.11174660,2-s2.0-105019047127
7,2025,Newmann Series-Based Precoding Weight Design f...,10.1109/VTC2025-Spring65109.2025.11174837,2-s2.0-105019051188
8,2025,Genetic Resource Allocation Algorithm for Pane...,10.3390/electronics14153107,None
9,2025,Genetic Resource Allocation Algorithm for Pane...,10.3390/electronics14153107,None
